In [15]:
import warnings
warnings.filterwarnings("ignore")

import os
import importlib

import pandas as pd

from sklearn.model_selection import StratifiedKFold

from config_models import (
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    OUTPUT_DIR,
)

import models_helpers
importlib.reload(models_helpers)

from models_helpers import (
    get_model_specs,
    tune_model,
    evaluate_tuned_models,
)
RANDOM_STATE = 42

## 0. Load Data

In [16]:
train_data = pd.read_csv(TRAIN_DATA_PATH)
test_data = pd.read_csv(TEST_DATA_PATH)

X_train = train_data.drop(columns=["patient_id", "hf_outcome"])
y_train = train_data["hf_outcome"]


X_test = test_data.drop(columns=["patient_id", "hf_outcome"])
y_test = test_data["hf_outcome"]

## 1. Parameters Tuning

In [17]:
model_specs = get_model_specs(
    y_train=y_train,
    random_state=RANDOM_STATE
)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [18]:
%%time
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = {}
best_params = {}
cv_summary_rows = []

for model_name, spec in model_specs.items():
    print(f"\nTuning {model_name}")

    search = tune_model(
        model_name=model_name,
        model=spec["model"],
        params=spec["params"],
        search_type=spec["search_type"],
        X_train=X_train,
        y_train=y_train,
        cv=cv,
        scoring="roc_auc",
        n_iter=spec["n_iter"],
        random_state=RANDOM_STATE,
    )

    results[model_name] = search #best hyperparameters search object

    best_params[model_name] = search.best_params_

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.to_csv(
        os.path.join(OUTPUT_DIR, f"cv_results_{model_name}.csv"),
        index=False,
    )

    cv_summary_rows.append({
        "model": model_name,
        "best_cv_score": search.best_score_,
        "best_params": str(search.best_params_),
    })

cv_summary = pd.DataFrame(cv_summary_rows)

cv_summary.to_csv(
    os.path.join(OUTPUT_DIR, "cv_summary.csv"),
    index=False,
)

best_params_df = (
    pd.DataFrame.from_dict(best_params, orient="index")
    .reset_index()
    .rename(columns={"index": "model"})
)
best_params_df.to_csv(
    os.path.join(OUTPUT_DIR, "best_params.csv"),
    index=False,
)

print("Saved CV results and best hyperparameters")


Tuning lr
Fitting 5 folds for each of 6 candidates, totalling 30 fits


lr: best CV roc_auc = 0.5835
lr: best params = {'model__C': 0.1, 'model__l1_ratio': 0.5, 'model__penalty': 'elasticnet', 'model__solver': 'saga'}

Tuning rf
Fitting 5 folds for each of 25 candidates, totalling 125 fits
rf: best CV roc_auc = 0.5900
rf: best params = {'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 10, 'criterion': 'entropy', 'class_weight': 'balanced_subsample'}

Tuning hgb
Fitting 5 folds for each of 20 candidates, totalling 100 fits
hgb: best CV roc_auc = 0.5772
hgb: best params = {'max_iter': 100, 'max_depth': 3, 'learning_rate': 0.1}

Tuning xgb
Fitting 5 folds for each of 25 candidates, totalling 125 fits
xgb: best CV roc_auc = 0.5904
xgb: best params = {'subsample': 0.7, 'scale_pos_weight': 0.24610591900311526, 'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.2}
Saved CV results and best hyperparameters
CPU times: user 3.5 s, sys: 1.77 s, total: 5.27 s
Wall time:

In [19]:
cv_summary.sort_values("best_cv_score", ascending=False)

,model,best_cv_score,best_params
3,xgb,0.590402,"{'subsample': 0.7, 'scale_pos_weight': 0.24610..."
1,rf,0.589964,"{'n_estimators': 150, 'min_samples_split': 10,..."
0,lr,0.583521,"{'model__C': 0.1, 'model__l1_ratio': 0.5, 'mod..."
2,hgb,0.577194,"{'max_iter': 100, 'max_depth': 3, 'learning_ra..."


## 3. Evaluate

In [21]:
results

{'lr': GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('imputer',
                                         SimpleImputer(add_indicator=True,
                                                       strategy='median')),
                                        ('scaler', StandardScaler()),
                                        ('model',
                                         LogisticRegression(class_weight='balanced',
                                                            max_iter=10000,
                                                            random_state=42))]),
              n_jobs=-1,
              param_grid=[{'model__C': [0.1, 1, 10], 'model__penalty': ['l2'],
                           'model__solver': ['saga']},
                          {'model__C': [0.1, 1, 10], 'model__l1_ratio': [0.5],
                           'model__penalty': ['elasticnet'],
                           'model__solver': ['saga']}

In [6]:
performance_df, prediction_df = evaluate_tuned_models(
    results=results,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    patient_ids=test_data["patient_id"],
    threshold=None,
)

performance_df.sort_values("auc", ascending=False)

Evaluating lr
Evaluating rf
Evaluating hgb
Evaluating xgb


,observed_positive_rate,observed_negative_rate,predicted_positive_rate,predicted_negative_rate,accuracy,precision,recall,f1,sensitivity,specificity,...,false_negatives,auc,auc_pr,brier,threshold,observed_event_rate,mean_predicted_risk,model,best_cv_score,threshold_method
3,0.803,0.197,0.800,0.200,0.713,0.822500,0.819427,0.820961,0.819427,0.279188,...,145,0.585131,0.841274,0.238355,0.462314,0.803,0.516698,xgb,0.590402,event_rate_train
1,0.803,0.197,0.850,0.150,0.745,0.822353,0.870486,0.845735,0.870486,0.233503,...,104,0.581443,0.841805,0.218882,0.475424,0.803,0.556233,rf,0.589964,event_rate_train
2,0.803,0.197,0.787,0.213,0.694,0.815756,0.799502,0.807547,0.799502,0.263959,...,161,0.581184,0.848116,0.157140,0.746854,0.803,0.802688,hgb,0.577194,event_rate_train
0,0.803,0.197,0.790,0.210,0.685,0.808861,0.795766,0.802260,0.795766,0.233503,...,164,0.566884,0.836822,0.247993,0.420861,0.803,0.512121,lr,0.583521,event_rate_train


## 4. Save

In [7]:
performance_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_performance.csv"),
    index=False,
)

prediction_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_predictions.csv.gz"),
    index=False,
    compression="gzip",
)

print("Saved test performance and predictions")

Saved test performance and predictions
